# Phase 10: Fold Robustness Validation

## 1. Packages and paths

In [7]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
from itertools import product
from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact

warnings.filterwarnings('default')
np.random.seed(36)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

#Paths
PROJECT_ROOT=Path.cwd().parent
PHASE6_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase6_models_folds'
PHASE10_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase10_fold_robustness'
PHASE10_OUTPUT.mkdir(parents=True, exist_ok=True)

#Constants
DATASET='2007'
FOLDS=['Fold1', 'Fold2', 'Fold3']
MODELS=['pointwise', 'pairwise', 'lightgbm']
PIPELINES=['raw', 'global', 'per_query']
EXPECTED_CONFIGS=len(MODELS) * len(PIPELINES)  # 9

BASELINE_MODEL='pointwise'
BASELINE_PIPELINE='raw'
MIN_SAMPLE_WARNING=10

warnings_issued=[]

print('='*80)
print('PHASE 10: FOLD ROBUSTNESS VALIDATION')
print('='*80)
print(f'Output: {PHASE10_OUTPUT}')
print(f'Dataset: MQ{DATASET}')
print(f'Folds: {FOLDS}')
print(f'Expected configs/fold: {EXPECTED_CONFIGS}')
print(f'Baseline: {BASELINE_MODEL}_{BASELINE_PIPELINE}')
print('='*80)

PHASE 10: FOLD ROBUSTNESS VALIDATION
Output: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase10_fold_robustness
Dataset: MQ2007
Folds: ['Fold1', 'Fold2', 'Fold3']
Expected configs/fold: 9
Baseline: pointwise_raw


## 2. Util functions

In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils import (
    make_fail_flag,
    check_sample_size,
    safe_k,
    compute_failure_at_k,
    mcnemar_test,
    bh_fdr,
)

In [9]:
def compute_score_gap(pred_df: pd.DataFrame, qids: set, k: int=5, relevance_threshold: int=1) -> dict:
    """
    Score gap=(best relevant score)-(score at rank k).
    Returns {qid: gap or np.nan}.
    """
    gaps={}
    for qid in qids:
        q_docs=pred_df[pred_df["qid"] == qid].copy()
        if len(q_docs)==0:
            gaps[qid]=np.nan
            continue

        q_docs=q_docs.sort_values("score", ascending=False)

        relevant=q_docs[q_docs["label"]>=relevance_threshold]
        if len(relevant)==0:
            gaps[qid]=np.nan
            continue

        best_rel=float(relevant["score"].max())

        k_actual=min(k, len(q_docs))
        if k_actual<=0:
            gaps[qid]=np.nan
            continue

        score_at_k=float(q_docs.iloc[k_actual-1]["score"])
        gaps[qid]=best_rel-score_at_k

    return gaps

print("imported src.utils + defined compute_score_gap")

imported src.utils + defined compute_score_gap


## 3. Loading Phase 6 Artifacts

In [10]:
print('\n'+'='*80)
print('LOADING PHASE 6 ARTIFACTS')
print('='*80)

all_fold_artifacts={}

for fold in FOLDS:
    print(f'\n{fold}:')
    artifacts={'query_metrics': {}, 'predictions': {}}
    
    for model, pipeline in product(MODELS, PIPELINES):
        key=f'{model}_{pipeline}_{DATASET}'
        qm_file=PHASE6_OUTPUT / fold/f'{key}_query_metrics.csv'
        pred_file=PHASE6_OUTPUT / fold/f'{key}_predictions.csv'
        
        if not qm_file.exists() or not pred_file.exists():
            print(f'Missing: {key}')
            continue
        
        qm=pd.read_csv(qm_file)
        pred=pd.read_csv(pred_file)
        
        #Proper validation with flag
        ok=True
        for col in ['qid', 'num_docs', 'num_relevant_1', 'Failure@5_primary']:
            if col not in qm.columns:
                print(f'{key} missing column: {col}')
                ok=False
        for col in ['qid', 'label', 'score']:
            if col not in pred.columns:
                print(f'{key} missing column: {col}')
                ok=False
        if not ok:
            continue
        
        #Coercing dtypes
        qm['qid']=qm['qid'].astype(int)
        pred['qid']=pred['qid'].astype(int)
        pred['label']=pred['label'].astype(int)
        pred['score']=pred['score'].astype(float)
        
        artifacts['query_metrics'][key]=qm
        artifacts['predictions'][key]=pred
    
    if len(artifacts['query_metrics'])>0:
        all_fold_artifacts[fold]=artifacts
        print(f'{len(artifacts["query_metrics"])} configs')
    else:
        msg=f'{fold}: no configs loaded'
        warnings_issued.append(msg)
        print(f'{msg}')

print(f'\nLoaded {len(all_fold_artifacts)}/{len(FOLDS)} folds')
print('='*80)


LOADING PHASE 6 ARTIFACTS

Fold1:
9 configs

Fold2:
9 configs

Fold3:
9 configs

Loaded 3/3 folds


## 4. Step 1: Baseline Failure Rates

In [11]:
print('\n'+'='*80)
print('STEP 1: BASELINE FAILURE RATES')
print('='*80)

baseline_rates=[]

for fold in FOLDS:
    if fold not in all_fold_artifacts:
        print(f'{fold}: skipped (not loaded)')
        continue
    
    baseline_key=f'{BASELINE_MODEL}_{BASELINE_PIPELINE}_{DATASET}'
    if baseline_key not in all_fold_artifacts[fold]['query_metrics']:
        msg=f'{fold}: missing {baseline_key}'
        warnings_issued.append(msg)
        print(f'{msg}')
        continue
    
    qm=all_fold_artifacts[fold]['query_metrics'][baseline_key]
    evaluable=qm[qm['num_relevant_1']>0]
    n_evaluable=len(evaluable)
    
    if n_evaluable==0:
        print(f'{fold}: no evaluable queries')
        continue
    
    fail_flag=make_fail_flag(evaluable['Failure@5_primary'])
    n_failures=int((fail_flag==1).sum())
    pct_failures=100 * n_failures / n_evaluable
    
    baseline_rates.append({
        'fold': fold,
        'n_evaluable': n_evaluable,
        'n_failures': n_failures,
        'pct_failures': float(pct_failures)
    })
    print(f'{fold}: {n_failures}/{n_evaluable} ({pct_failures:.1f}%)')

baseline_df=pd.DataFrame(baseline_rates).sort_values('fold')
if len(baseline_df)>0:
    display(baseline_df)
    baseline_df.to_csv(PHASE10_OUTPUT / 'phase10_fold_failure_rates.csv', index=False)
    print('\nSaved: phase10_fold_failure_rates.csv')
else:
    baseline_df = pd.DataFrame()
    print('No baseline rates computed')


STEP 1: BASELINE FAILURE RATES
Fold1: 44/290 (15.2%)
Fold2: 39/283 (13.8%)
Fold3: 59/295 (20.0%)


,fold,n_evaluable,n_failures,pct_failures
0,Fold1,290,44,15.1724
1,Fold2,283,39,13.7809
2,Fold3,295,59,20.0000



Saved: phase10_fold_failure_rates.csv


Here we are looking at how the baseline model performs on each fold separately. We are only counting *evaluable* queries, meaning queries where there was at least one relevant document somewhere in the list. So these are the cases where the model actually had a fair chance to succeed.

What this tells us is that the baseline model fails on roughly **14–20%** of evaluable queries depending on the fold. The numbers are not identical, which is expected because each fold contains slightly different training and test splits. 

Fold3 seems a bit harder since it has a higher failure rate, while Fold2 looks slightly easier. But overall, the failure rates are in a similar range. There is some variation, but nothing extreme or suspicious.

This gives us a stable starting point. Now we know how often the baseline fails in each fold, and we can later check whether persistent failures are consistent across folds or just due to random split differences.

## 5. Step 2: Persistent Queries

In [13]:
print('\n'+'='*80)
print('STEP 2: PERSISTENT QUERIES')
print('='*80)
print('Persistent = intersection of failures across ALL 9 configs')
print('='*80)

persistent_results=[]
fold_persistent_qids={}

for fold in FOLDS:
    if fold not in all_fold_artifacts:
        print(f'{fold}: skipped')
        continue
    
    qm_dict=all_fold_artifacts[fold]['query_metrics']
    
    #Config count guard
    if len(qm_dict) != EXPECTED_CONFIGS:
        msg=f'{fold}: {len(qm_dict)}/{EXPECTED_CONFIGS} configs -> skip persistent'
        warnings_issued.append(msg)
        print(f'{msg}')
        continue
    
    evaluable_sets={}
    failure_sets={}
    
    for key, qm in qm_dict.items():
        evaluable_sets[key]=set(qm.loc[qm['num_relevant_1'] > 0, 'qid'])
        fail_flag=make_fail_flag(qm['Failure@5_primary'])
        fail_mask=(qm['num_relevant_1']>0) & (fail_flag==1)
        failure_sets[key]=set(qm.loc[fail_mask, 'qid'])
    
    #Empty guards
    if len(evaluable_sets)==0 or len(failure_sets)==0:
        msg=f'{fold}: empty sets'
        warnings_issued.append(msg)
        print(f'{msg}')
        continue
    
    all_evaluable=set.intersection(*evaluable_sets.values())
    persistent=set.intersection(*failure_sets.values())
    all_failing=set.union(*failure_sets.values())
    successful=all_evaluable - all_failing
    
    pct_fail=100 * len(persistent) / len(all_failing) if len(all_failing) > 0 else 0
    pct_eval=100 * len(persistent) / len(all_evaluable) if len(all_evaluable) > 0 else 0
    
    check_sample_size(f'{fold} persistent', len(persistent))
    check_sample_size(f'{fold} successful', len(successful))
    
    persistent_results.append({
        'fold': fold,
        'n_evaluable': len(all_evaluable),
        'n_failing': len(all_failing),
        'n_persistent': len(persistent),
        'n_successful': len(successful),
        'pct_persistent_of_failing': float(pct_fail),
        'pct_persistent_of_evaluable': float(pct_eval)
    })
    
    fold_persistent_qids[fold] = {
        'persistent': persistent,
        'successful': successful
    }
    print(f'{fold}: {len(persistent)}/{len(all_evaluable)} ({pct_eval:.1f}%)')

persistent_df=pd.DataFrame(persistent_results).sort_values('fold')
if len(persistent_df) > 0:
    display(persistent_df)
    persistent_df.to_csv(PHASE10_OUTPUT / 'phase10_fold_persistent_summary.csv', index=False)
    print('\nSaved: phase10_fold_persistent_summary.csv')
else:
    persistent_df=pd.DataFrame()
    print('No persistent data')


STEP 2: PERSISTENT QUERIES
Persistent = intersection of failures across ALL 9 configs
Fold1: 22/290 (7.6%)
Fold2: 20/283 (7.1%)
Fold3: 29/295 (9.8%)


,fold,n_evaluable,n_failing,n_persistent,n_successful,pct_persistent_of_failing,pct_persistent_of_evaluable
0,Fold1,290,85,22,205,25.8824,7.5862
1,Fold2,283,75,20,208,26.6667,7.0671
2,Fold3,295,84,29,211,34.5238,9.8305



Saved: phase10_fold_persistent_summary.csv


So across all folds, persistent failures are in the same general range, around **7–10%** of evaluable queries. Fold3 is a bit higher, but it’s not like a totally different story.

About **a quarter to one-third** of the queries that fail at least once are actually persistent failures. That’s kind of important because it suggests a chunk of the failures are not just random or caused by one specific model choice.

Overall, this looks pretty consistent across folds. It supports the idea that persistent failures are a real thing in every fold, not just Fold1 being lucky.


## 6. Step 3: Structural Direction

In [14]:
print('\n'+'='*80)
print('STEP 3: STRUCTURAL DIRECTION')
print('='*80)
print('Expected based on Phase 8 Fold1 findings:')
print('  persistent < successful (num_relevant_1)')
print('  persistent > successful (sparsity)')
print('  persistent < successful (score_gap)')
print('='*80)

structural_results=[]

for fold in FOLDS:
    if fold not in all_fold_artifacts or fold not in fold_persistent_qids:
        continue
    
    baseline_key=f'{BASELINE_MODEL}_{BASELINE_PIPELINE}_{DATASET}'
    if baseline_key not in all_fold_artifacts[fold]['query_metrics']:
        continue
    
    print(f'\n{fold}:')
    qm=all_fold_artifacts[fold]['query_metrics'][baseline_key]
    pred=all_fold_artifacts[fold]['predictions'][baseline_key]
    
    pers=qm[qm['qid'].isin(fold_persistent_qids[fold]['persistent'])]
    succ=qm[qm['qid'].isin(fold_persistent_qids[fold]['successful'])]
    
    if len(pers)==0 or len(succ)==0:
        print('insufficient data')
        continue
    
    #num_relevant_1
    pers_rel=pers['num_relevant_1'].values
    succ_rel=succ['num_relevant_1'].values
    try:
        _, pval=mannwhitneyu(pers_rel, succ_rel, alternative='two-sided')
    except:
        pval=1.0
    med_p=float(np.median(pers_rel))
    med_s=float(np.median(succ_rel))
    consistent=med_p<med_s
    structural_results.append({
        'fold': fold, 'metric': 'num_relevant_1',
        'median_persistent': med_p, 'median_successful': med_s,
        'pval': float(pval), 'direction_consistent': consistent,
        'expected_direction': 'persistent < successful'
    })
    
    check='Yes' if consistent else 'No'
    print(f'num_rel: p={med_p:.2f} s={med_s:.2f} pv={pval:.4f} {check}')
    
    #sparsity
    p_sp=(pers['num_relevant_1']==1).sum()
    s_sp=(succ['num_relevant_1']==1).sum()
    rate_p=p_sp / len(pers)
    rate_s=s_sp / len(succ)
    consistent2=rate_p > rate_s
    cont=[[p_sp, len(pers)-p_sp], [s_sp, len(succ)-s_sp]]
    if min(p_sp, len(pers)-p_sp, s_sp, len(succ)-s_sp) >= 5:
        _, pval, _, _ = chi2_contingency(cont)
    else:
        _, pval=fisher_exact(cont)
    structural_results.append({
        'fold': fold, 'metric': 'pct_num_rel_eq_1',
        'median_persistent': float(rate_p*100), 'median_successful': float(rate_s*100),
        'pval': float(pval), 'direction_consistent': consistent2,
        'expected_direction': 'persistent > successful'
    })
    check2 = 'Yes' if consistent2 else 'No'
    print(f'sparsity: p={rate_p*100:.1f}% s={rate_s*100:.1f}% pv={pval:.4f} {check2}')
    
    #score_gap
    gaps=compute_score_gap(pred, fold_persistent_qids[fold]['persistent'] | fold_persistent_qids[fold]['successful'])
    p_gaps=[gaps[q] for q in fold_persistent_qids[fold]['persistent'] if q in gaps and not np.isnan(gaps[q])]
    s_gaps=[gaps[q] for q in fold_persistent_qids[fold]['successful'] if q in gaps and not np.isnan(gaps[q])]
    if len(p_gaps)>=6 and len(s_gaps)>=6:
        try:
            _, pval=mannwhitneyu(p_gaps, s_gaps, alternative='two-sided')
        except:
            pval=1.0
        med_pg=float(np.median(p_gaps))
        med_sg=float(np.median(s_gaps))
        consistent3=med_pg < med_sg
        structural_results.append({
            'fold': fold, 'metric': 'score_gap',
            'median_persistent': med_pg, 'median_successful': med_sg,
            'pval': float(pval), 'direction_consistent': consistent3,
            'expected_direction': 'persistent < successful'
        })
        check3 = 'Yes' if consistent3 else 'No'
        print(f'gap: p={med_pg:.3f} s={med_sg:.3f} pv={pval:.4f} {check3}')

structural_df=pd.DataFrame(structural_results).sort_values(['fold','metric'])
if len(structural_df)>0:
    display(structural_df[['fold','metric','direction_consistent','pval']])
    structural_df.to_csv(PHASE10_OUTPUT / 'phase10_fold_structural_direction.csv', index=False)
    print('\nSaved: phase10_fold_structural_direction.csv')
else:
    structural_df = pd.DataFrame()
    print('No structural tests')


STEP 3: STRUCTURAL DIRECTION
Expected based on Phase 8 Fold1 findings:
  persistent < successful (num_relevant_1)
  persistent > successful (sparsity)
  persistent < successful (score_gap)

Fold1:
num_rel: p=1.50 s=16.00 pv=0.0000 Yes
sparsity: p=50.0% s=4.9% pv=0.0000 Yes
gap: p=-0.156 s=0.118 pv=0.0000 Yes

Fold2:
num_rel: p=2.00 s=14.00 pv=0.0000 Yes
sparsity: p=40.0% s=1.9% pv=0.0000 Yes
gap: p=-0.104 s=0.118 pv=0.0000 Yes

Fold3:
num_rel: p=2.00 s=12.00 pv=0.0000 Yes
sparsity: p=27.6% s=5.2% pv=0.0001 Yes
gap: p=-0.151 s=0.088 pv=0.0000 Yes


,fold,metric,direction_consistent,pval
0,Fold1,num_relevant_1,True,0.0000
1,Fold1,pct_num_rel_eq_1,True,0.0000
2,Fold1,score_gap,True,0.0000
3,Fold2,num_relevant_1,True,0.0000
4,Fold2,pct_num_rel_eq_1,True,0.0000
5,Fold2,score_gap,True,0.0000
6,Fold3,num_relevant_1,True,0.0000
7,Fold3,pct_num_rel_eq_1,True,0.0001
8,Fold3,score_gap,True,0.0000



Saved: phase10_fold_structural_direction.csv


## 7. Step 4: Model Direction

In [15]:
print('\n'+'='*80)
print('STEP 4: MODEL DIRECTION (LightGBM vs Baseline)')
print('='*80)
print('Using common evaluable qids')
print('='*80)

model_results=[]

for fold in FOLDS:
    if fold not in all_fold_artifacts:
        continue
    baseline_key=f'{BASELINE_MODEL}_{BASELINE_PIPELINE}_{DATASET}'
    lgbm_key=f'lightgbm_per_query_{DATASET}'
    if baseline_key not in all_fold_artifacts[fold]['query_metrics']:
        continue
    if lgbm_key not in all_fold_artifacts[fold]['query_metrics']:
        print(f'{fold}: no LightGBM')
        continue
    
    qm_b=all_fold_artifacts[fold]['query_metrics'][baseline_key]
    qm_l=all_fold_artifacts[fold]['query_metrics'][lgbm_key]
    
    #Common qids
    eval_b=set(qm_b.loc[qm_b['num_relevant_1'] > 0, 'qid'])
    eval_l=set(qm_l.loc[qm_l['num_relevant_1'] > 0, 'qid'])
    common=eval_b & eval_l
    if len(common)<10:
        print(f'{fold}: insufficient common ({len(common)})')
        continue
    
    qm_b_c=qm_b[qm_b['qid'].isin(common)]
    qm_l_c=qm_l[qm_l['qid'].isin(common)]
    fail_b=make_fail_flag(qm_b_c['Failure@5_primary'])
    fail_l=make_fail_flag(qm_l_c['Failure@5_primary'])
    n_b=int((fail_b == 1).sum())
    n_l=int((fail_l == 1).sum())
    pct_b=100 * n_b / len(common)
    pct_l=100 * n_l / len(common)
    improved=pct_l < pct_b
    model_results.append({
        'fold': fold, 'n_common_evaluable': len(common),
        'baseline_pct': float(pct_b), 'lightgbm_pct': float(pct_l),
        'direction_improved': improved
    })
    check = 'Yes' if improved else 'No'
    print(f'{fold}: b={pct_b:.1f}% l={pct_l:.1f}% {check}')

model_df=pd.DataFrame(model_results).sort_values('fold')
if len(model_df) > 0:
    display(model_df)
    model_df.to_csv(PHASE10_OUTPUT / 'phase10_fold_model_direction.csv', index=False)
    print('\nSaved: phase10_fold_model_direction.csv')
else:
    model_df=pd.DataFrame()
    print('No model comparisons')


STEP 4: MODEL DIRECTION (LightGBM vs Baseline)
Using common evaluable qids
Fold1: b=15.2% l=15.2% No
Fold2: b=13.8% l=16.6% No
Fold3: b=20.0% l=18.6% Yes


,fold,n_common_evaluable,baseline_pct,lightgbm_pct,direction_improved
0,Fold1,290,15.1724,15.1724,False
1,Fold2,283,13.7809,16.6078,False
2,Fold3,295,20.0000,18.6441,True



Saved: phase10_fold_model_direction.csv


## 8. Summary JSON

In [ ]:
print('\n'+'='*80)
print('SUMMARY JSON')
print('='*80)

#Direction consistency
direction_consistency={}
if len(structural_df)>0:
    for m in structural_df['metric'].unique():
        all_ok = structural_df[structural_df['metric']==m]['direction_consistent'].all()
        direction_consistency[m]=bool(all_ok)

#Baseline rates
baseline_failure_rates={}
if len(baseline_df)>0:
    for _, r in baseline_df.iterrows():
        baseline_failure_rates[r['fold']]=float(r['pct_failures'])

#Persistent rates
persistent_rates={}
if len(persistent_df)>0:
    for _, r in persistent_df.iterrows():
        persistent_rates[r['fold']]=float(r['pct_persistent_of_evaluable'])

summary={
    'phase': 10,
    'objective': 'Fold robustness validation',
    'dataset': f'MQ{DATASET}',
    'folds_tested': FOLDS,
    'expected_configs_per_fold': EXPECTED_CONFIGS,
    'baseline_failure_rates': baseline_failure_rates,
    'persistent_rates': persistent_rates,
    'direction_consistency': direction_consistency,
    'n_structural_tests': len(structural_df),
    'n_model_comparisons': len(model_df),
    'warnings': warnings_issued
}

with open(PHASE10_OUTPUT / 'phase10_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print('\nSaved: phase10_summary.json')